# Clase 6 — Audio, NLP y cómo funciona un transformer

## Pregunta central

> **¿Cómo pasan una señal de audio y una frase a ser datos que un modelo puede procesar?**

## Idea principal

Audio y texto se convierten en números; los modelos encuentran patrones en esas representaciones y producen una salida.

## Objetivos de aprendizaje

Al finalizar la clase deberías poder:

- Leer una señal de audio y reconocer sample rate, waveform y espectrograma.
- Explicar qué hacen ASR y TTS, y ejecutar una transcripción pequeña.
- Ubicar tokenización, clasificación de texto y NER dentro de un pipeline de NLP.
- Explicar intuitivamente Query, Key, Value y attention.
- Relacionar contexto, generación autoregresiva, velocidad y memoria de la KV cache.

## Recorrido de la clase

| Paso | Tema |
|---:|---|
| 1 | Del sonido a los números |
| 2 | Espectrogramas y features acústicas |
| 3 | ASR y TTS |
| 4 | Del texto a los tokens |
| 5 | Transformers y attention |
| 6 | Contexto, generación y KV cache |
| 7 | Actividad integradora |

## Cómo trabajar con este notebook

1. Ejecutá las celdas en el orden propuesto.
2. Antes de modificar código, observá y describí el resultado.
3. Cambiá solamente las variables marcadas con `TODO`.
4. No es necesario implementar algoritmos desde cero.
5. Si aparece un término nuevo, buscá primero su definición en el glosario de la clase.

**Conexión con el programa:** las clases de audio, NLP, LLMs y agentes del Track Salud.

## Glosario mínimo y mapa de la clase

En esta clase conectaremos dos tipos de entrada:

```text
AUDIO
  sonido → muestras numéricas → waveform → espectrograma
                                          ↓
                                         ASR → texto

TEXTO
  texto → tokens → embeddings → transformer → tokens de salida
                                ├── attention
                                └── KV cache durante la generación
```

Primero necesitamos un vocabulario común. No hace falta memorizar los
nombres en inglés: lo importante es poder explicar qué entra, qué
transformación ocurre y qué sale.

### Audio

| Término | Definición para esta clase |
|---|---|
| Señal | Una cantidad que cambia en el tiempo |
| Muestra | Una medición numérica de la señal en un instante |
| Waveform | Secuencia de amplitudes de una señal a lo largo del tiempo |
| Sample rate | Cantidad de muestras capturadas por segundo |
| Frecuencia | Cantidad de ciclos por segundo; se mide en hertz (Hz) |
| Espectrograma | Tabla o imagen que muestra energía por tiempo y frecuencia |
| dB o decibel | Escala logarítmica usada para expresar niveles relativos de energía |
| Feature | Valor calculado para resumir alguna propiedad del dato |
| ASR | Sistema que convierte voz o habla en texto |
| TTS | Sistema que convierte texto en una señal de voz |

### Lenguaje y modelos

| Término | Definición para esta clase |
|---|---|
| NLP o PLN | Procesamiento de Lenguaje Natural: técnicas para trabajar con lenguaje mediante software |
| Token | Unidad en la que un tokenizador divide el texto |
| Prompt | Secuencia inicial de instrucciones, contexto y datos entregada al modelo |
| Embedding | Vector numérico que representa un token u otro contenido |
| Transformer | Arquitectura de red neuronal basada en bloques de attention |
| Attention | Mecanismo que asigna pesos para combinar información del contexto |
| LLM | Modelo de lenguaje de gran escala que estima continuaciones de tokens |
| Ventana de contexto | Cantidad máxima de tokens disponibles durante una inferencia |
| KV cache | Keys y Values ya calculados que se reutilizan durante la generación |

### Distribución sugerida de las dos horas

| Bloque | Minutos |
|---|---:|
| Mapa inicial | 10 |
| Waveform y lectura del WAV | 15 |
| Espectrograma y features | 15 |
| ASR y TTS | 15 |
| Tokens y tareas de NLP | 15 |
| Transformer y attention | 20 |
| Generación y KV cache | 10 |
| Actividad integradora | 20 |

---
## 1. Del sonido a los números

Cuando hablamos, producimos variaciones de presión en el aire. Un
micrófono transforma esas variaciones en una señal eléctrica y un
conversor toma mediciones a intervalos regulares.

```text
presión del aire
      ↓ micrófono
señal eléctrica continua
      ↓ muestreo
[medición 0, medición 1, medición 2, ...]
      ↓ archivo WAV
secuencia numérica que puede leer Python
```

La **amplitud** es el valor de una muestra respecto del punto de
equilibrio de la señal. En el gráfico será el eje vertical. La
waveform se parece a una serie temporal de telemetría: cada posición
tiene un instante conocido y un valor.

| Concepto | Qué significa | Ejemplo de esta clase |
|---|---|---|
| Muestra | Una medición individual | Un número entero de 16 bits |
| Sample rate | Mediciones tomadas por segundo | 16.000 Hz o 16 kHz |
| Canal | Una secuencia independiente | Mono: un canal |
| PCM | Forma de guardar directamente valores de muestras | WAV PCM |
| Duración | `cantidad de muestras / sample rate` | Aproximadamente 3,7 s |
| Normalización | Cambio de escala numérica | Enteros → valores cercanos a `[-1, 1]` |

`kHz` significa miles de hertz: `16 kHz = 16.000 Hz`. Con este sample
rate podemos representar frecuencias de hasta aproximadamente 8 kHz.
Esa mitad surge del principio de muestreo y explica el límite vertical
del espectrograma que veremos después.

Un sample rate más alto permite conservar frecuencias más altas, pero
también produce más datos. No recupera información que ya se perdió
durante una grabación de menor calidad. Normalizar tampoco agrega
información: solo coloca las amplitudes en una escala cómoda para el
cálculo.

### El audio de esta clase

`assets/audio/curso_ia_es.wav` fue generado especialmente para este
curso con una voz sintética en español y convertido a WAV PCM mono de
16 kHz. No contiene voz ni datos personales de ninguna persona.

**Transcripción de referencia:** “La inteligencia artificial aprende
patrones a partir de datos.”

In [ ]:
from pathlib import Path
import os
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from scipy.io import wavfile
from scipy.signal import spectrogram

FAST_MODE = True
SEED = 42
rng = np.random.default_rng(SEED)

TRANSCRIPCION_REFERENCIA = (
    "La inteligencia artificial aprende patrones a partir de datos."
)


def encontrar_asset(ruta_relativa):
    """Encuentra el asset al ejecutar desde el curso o desde el repo."""
    ruta_relativa = Path(ruta_relativa)
    candidatos = [
        Path.cwd() / ruta_relativa,
        Path.cwd() / "IA para programadores" / ruta_relativa,
    ]
    for padre in Path.cwd().parents:
        candidatos.append(
            padre / "IA para programadores" / ruta_relativa
        )
    for candidato in candidatos:
        if candidato.exists():
            return candidato.resolve()
    raise FileNotFoundError(
        f"No se encontró {ruta_relativa}. "
        "Ejecutá el notebook desde el directorio del curso o del repo."
    )


ruta_audio = encontrar_asset("assets/audio/curso_ia_es.wav")
sample_rate, audio_pcm = wavfile.read(ruta_audio)

# Normalizamos PCM entero al rango aproximado [-1, 1].
if np.issubdtype(audio_pcm.dtype, np.integer):
    limite = max(abs(np.iinfo(audio_pcm.dtype).min), np.iinfo(audio_pcm.dtype).max)
    audio = audio_pcm.astype(np.float32) / limite
else:
    audio = audio_pcm.astype(np.float32)

if audio.ndim == 2:
    audio = audio.mean(axis=1)

duracion = len(audio) / sample_rate
resumen_audio = pd.DataFrame(
    {
        "propiedad": [
            "archivo", "sample rate", "canales",
            "muestras", "duración", "dtype original",
        ],
        "valor": [
            ruta_audio.name,
            f"{sample_rate:,} Hz",
            "1 (mono)",
            f"{len(audio):,}",
            f"{duracion:.2f} s",
            str(audio_pcm.dtype),
        ],
    }
)
display(resumen_audio)
display(Audio(audio, rate=sample_rate))

El reproductor permite escuchar la misma información que veremos en
un gráfico. Para una computadora todavía no hay palabras: solo una
secuencia de amplitudes y el dato de cuántas muestras corresponden a
un segundo.

La tabla confirma el **contrato de entrada** del pipeline:

```text
archivo WAV + sample rate → arreglo de una dimensión → modelo de audio
```

Si el archivo tuviera dos canales, el arreglo tendría una segunda
dimensión. El código los promediaría para obtener mono. La
normalización conserva la forma de la onda, pero cambia su escala.

La **energía RMS** que imprime la próxima celda es una medida resumida
de la amplitud. RMS significa raíz del promedio de los cuadrados. Un
valor mayor suele acompañar una señal más intensa, aunque no equivale
exactamente a la percepción humana de volumen.

In [ ]:
tiempo = np.arange(len(audio)) / sample_rate

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.plot(tiempo, audio, color="#2563eb", linewidth=0.7)
ax.set(
    title="Waveform: amplitud a lo largo del tiempo",
    xlabel="Tiempo (segundos)",
    ylabel="Amplitud normalizada",
)
ax.axhline(0, color="black", linewidth=0.6)
ax.grid(alpha=0.2)
plt.show()

print(f"Máxima amplitud absoluta: {np.max(np.abs(audio)):.3f}")
print(f"Energía RMS: {np.sqrt(np.mean(audio ** 2)):.3f}")

### Qué observar

- Los bloques con mayor amplitud suelen coincidir con sonido.
- Los espacios de baja amplitud pueden ser pausas, pero amplitud baja
  no equivale siempre a silencio.
- La waveform muestra **cuándo** cambia la señal; no muestra con
  claridad **qué frecuencias** contiene.
- Dos palabras diferentes pueden producir waveforms visualmente
  parecidas. El gráfico por sí solo no transcribe el contenido.
- Un pico aislado puede ser ruido. El modelo necesita patrones a lo
  largo de muchas muestras, no una única amplitud.

**Pregunta de control:** si duplicamos el sample rate declarado sin
agregar muestras, ¿la frase dura lo mismo? No: se reproduciría en la
mitad del tiempo. El sample rate forma parte del significado temporal
del arreglo.

---
## 2. Espectrogramas y features acústicas

La **frecuencia** describe cuántas veces se repite una oscilación por
segundo. Un tono simple concentra energía cerca de una frecuencia; la
voz combina muchas frecuencias que cambian rápidamente.

Un único análisis de toda la grabación diría qué frecuencias existen,
pero perdería cuándo aparecen. Para conservar ambas perspectivas, el
espectrograma divide el audio en **ventanas**, es decir, tramos cortos
y parcialmente superpuestos.

```text
waveform
  ├── ventana 1 → energía por frecuencia
  ├── ventana 2 → energía por frecuencia
  ├── ventana 3 → energía por frecuencia
  └── ...       → apilar columnas → espectrograma
```

El **espectrograma** resume qué frecuencias tienen energía en cada
instante:

- eje horizontal: tiempo;
- eje vertical: frecuencia;
- color: energía.

El gráfico expresa energía relativa en **decibeles (dB)**. Es una
escala logarítmica: comprime diferencias muy grandes para poder
visualizarlas. En este notebook importa comparar colores dentro del
mismo gráfico, no interpretar un dB como una transcripción.

El tamaño de ventana define un intercambio:

| Ventana | Ventaja | Costo |
|---|---|---|
| Más corta | Ubica mejor cambios rápidos | Distingue peor frecuencias cercanas |
| Más larga | Distingue mejor frecuencias cercanas | Ubica peor cambios rápidos |

Una **feature acústica** es un número calculado a partir del audio para
resumir una propiedad. Las que veremos son:

| Feature | Qué resume |
|---|---|
| Duración | Tiempo total de la grabación |
| Energía RMS | Nivel medio de amplitud |
| Pico absoluto | Mayor amplitud encontrada |
| Tasa de cruces por cero | Frecuencia con que la señal cambia de signo |

Modelos de voz también pueden recibir un **espectrograma log-Mel**:
una versión que agrupa frecuencias en bandas y comprime la escala de
energía. “Mel” es una escala diseñada para aproximar cómo distinguimos
frecuencias; “log” comprime diferencias muy grandes de energía. En
esta nivelación no lo implementaremos.

> El espectrograma se dibuja como imagen, pero no es una fotografía
> RGB. Sus ejes y sus valores tienen significado acústico.

In [ ]:
frecuencias, instantes, potencia = spectrogram(
    audio,
    fs=sample_rate,
    window="hann",
    nperseg=400,   # 25 ms a 16 kHz
    noverlap=240,
    nfft=512,
    mode="psd",
)
potencia_db = 10 * np.log10(potencia + 1e-10)

fig, ax = plt.subplots(figsize=(12, 4.5))
imagen = ax.pcolormesh(
    instantes,
    frecuencias,
    potencia_db,
    shading="auto",
    cmap="magma",
)
ax.set(
    title="Espectrograma del audio de ejemplo",
    xlabel="Tiempo (segundos)",
    ylabel="Frecuencia (Hz)",
    ylim=(0, 8000),
)
fig.colorbar(imagen, ax=ax, label="Energía (dB)")
plt.show()

cruces_por_cero = np.mean(np.signbit(audio[1:]) != np.signbit(audio[:-1]))
features_simples = pd.Series(
    {
        "duración_s": duracion,
        "energía_rms": float(np.sqrt(np.mean(audio ** 2))),
        "pico_absoluto": float(np.max(np.abs(audio))),
        "tasa_cruces_por_cero": float(cruces_por_cero),
    },
    name="valor",
)
display(features_simples.round(4).to_frame())

### Cómo leer la salida

El espectrograma permite relacionar regiones de la waveform con
cambios de frecuencia. Las zonas más claras tienen más energía en esa
combinación de tiempo y frecuencia.

Las cuatro features de la tabla comprimen todo el archivo en cuatro
números. Esa compresión facilita comparar aspectos generales, pero
elimina el orden detallado:

```text
waveform completa: decenas de miles de valores ordenados
                      ↓ resumen
cuatro features:     duración, RMS, pico, cruces por cero
```

Dos grabaciones con las mismas cuatro features pueden contener frases
completamente diferentes. Por eso un sistema de reconocimiento de voz
conserva una representación temporal mucho más rica.

**Idea clave:** preprocesar no significa necesariamente “achicar”.
Significa transformar los datos a una representación coherente con la
tarea y con lo que espera el modelo.

---
## 3. ASR y TTS: dos direcciones diferentes

**ASR** significa *Automatic Speech Recognition* o reconocimiento
automático del habla. Su objetivo es producir texto a partir de una
señal de voz. **TTS** significa *Text to Speech*: produce una señal de
voz a partir de texto.

| Tarea | Entrada | Salida | Ejemplo |
|---|---|---|---|
| **ASR** (Automatic Speech Recognition) | Audio | Texto | Transcribir una consulta |
| **TTS** (Text to Speech) | Texto | Audio | Leer una respuesta |

Son tareas inversas en la forma de entrada y salida, pero no son el
mismo modelo ejecutado al revés.

```text
ASR: WAV → representación acústica → tokens estimados → texto
TTS: texto → representación lingüística → señal acústica → WAV
```

Un **modelo** es una función cuyos parámetros fueron ajustados usando
datos. Una **inferencia** es usar esos parámetros ya entrenados para
procesar una entrada nueva. No entrenaremos Whisper: solo haremos una
inferencia.

Un pipeline de ASR suele seguir estos pasos:

1. leer y normalizar el audio;
2. construir una representación acústica;
3. el modelo estima identificadores de tokens;
4. un decodificador convierte esos identificadores en texto.

El término **decodificar** aquí significa transformar una
representación interna en una salida legible. No implica descifrar
contenido cifrado.

### Qué hace y qué no hace ASR

| Sí hace | No garantiza |
|---|---|
| Convierte patrones acústicos en texto probable | Comprender la intención |
| Conserva el orden de lo dicho | Distinguir siempre nombres técnicos |
| Puede incluir puntuación | Veracidad del contenido hablado |
| Puede trabajar con varios idiomas | Exactitud en cualquier micrófono o ambiente |

La práctica intenta ejecutar `openai/whisper-tiny`, un modelo
multilingüe pequeño. La primera ejecución lo descarga y las siguientes
lo leen de **cache**, un almacenamiento local que evita repetir la
descarga. Para ensayar el aula sin red se puede iniciar
Jupyter con `CURSO_DESCARGAR_MODELOS=0`; si el modelo tampoco está
cacheado, se muestra la transcripción de referencia y se marca
explícitamente que se usó el fallback.

Un **fallback** es un camino alternativo para que el material siga
funcionando cuando falta un recurso. En este notebook devuelve un
texto conocido; no ejecuta un segundo modelo.

> El fallback permite continuar la explicación; **no cuenta como una
> inferencia ni sirve para medir la calidad de Whisper**.

In [ ]:
MODELO_ASR = "openai/whisper-tiny"
PERMITIR_DESCARGA = os.getenv("CURSO_DESCARGAR_MODELOS", "1") == "1"


def transcribir_whisper(
    señal,
    frecuencia_muestreo,
    permitir_descarga=False,
):
    """Transcribe con Whisper Tiny o devuelve un fallback explícito."""
    inicio = time.perf_counter()
    transformers_logging = None
    nivel_logging_anterior = None
    try:
        import torch
        from transformers import (
            AutoModelForSpeechSeq2Seq,
            AutoProcessor,
        )
        from transformers.utils import logging as transformers_logging

        # Evita que advertencias internas de versiones específicas
        # oculten el resultado pedagógico de la celda.
        nivel_logging_anterior = transformers_logging.get_verbosity()
        transformers_logging.set_verbosity_error()

        solo_local = not permitir_descarga
        processor = AutoProcessor.from_pretrained(
            MODELO_ASR,
            local_files_only=solo_local,
        )
        model = AutoModelForSpeechSeq2Seq.from_pretrained(
            MODELO_ASR,
            local_files_only=solo_local,
            torch_dtype=torch.float32,
            low_cpu_mem_usage=True,
        )
        model.eval()
        torch.set_num_threads(min(4, torch.get_num_threads()))

        entradas = processor(
            señal,
            sampling_rate=frecuencia_muestreo,
            return_tensors="pt",
            return_attention_mask=True,
        )
        argumentos_generacion = {
            "input_features": entradas.input_features,
            "language": "spanish",
            "task": "transcribe",
            "max_new_tokens": 48 if FAST_MODE else 96,
        }
        if hasattr(entradas, "attention_mask"):
            argumentos_generacion["attention_mask"] = (
                entradas.attention_mask
            )
        with torch.inference_mode():
            ids = model.generate(**argumentos_generacion)
        texto = processor.batch_decode(
            ids,
            skip_special_tokens=True,
        )[0].strip()
        modo = "Whisper Tiny (inferencia real)"
        detalle = "modelo cargado desde cache" if solo_local else (
            "cache local o descarga habilitada"
        )
    except Exception as error:
        texto = TRANSCRIPCION_REFERENCIA
        modo = "fallback precomputado"
        detalle = f"{type(error).__name__}: {str(error).splitlines()[0][:180]}"
    finally:
        if (
            transformers_logging is not None
            and nivel_logging_anterior is not None
        ):
            transformers_logging.set_verbosity(
                nivel_logging_anterior
            )

    return {
        "texto": texto,
        "modo": modo,
        "detalle": detalle,
        "segundos": time.perf_counter() - inicio,
    }


resultado_asr = transcribir_whisper(
    audio,
    sample_rate,
    permitir_descarga=PERMITIR_DESCARGA,
)

print("Modo:", resultado_asr["modo"])
print("Texto:", resultado_asr["texto"])
print(f"Tiempo: {resultado_asr['segundos']:.2f} s")
print("Detalle:", resultado_asr["detalle"])

if resultado_asr["modo"].startswith("fallback"):
    print(
        "\nPara preparar Whisper antes de la clase, habilitá internet "
        "y ejecutá nuevamente esta celda."
    )

### Interpretar el resultado

Una transcripción puede cambiar mayúsculas o puntuación sin cambiar
su significado. Por eso una comparación exacta de strings es una
verificación demasiado rígida.

Una métrica frecuente es **WER** (*Word Error Rate* o tasa de error de
palabras). Cuenta sustituciones, palabras omitidas y palabras
agregadas respecto de una transcripción de referencia. Una WER menor
es mejor, pero una única grabación no alcanza para estimar calidad.

En proyectos reales se evalúa sobre muchas grabaciones
representativas de:

- micrófonos y canales reales;
- ruido y distancia;
- acentos y velocidades de habla;
- vocabulario técnico;
- silencios, interrupciones y audios vacíos.

Whisper no “escucha” como una persona: estima una secuencia de texto
probable según la señal y los patrones aprendidos durante su
entrenamiento.

### TTS, solo como mapa

En TTS la dirección se invierte: texto → representación lingüística
→ representación acústica → waveform. La **prosodia** reúne aspectos
como ritmo, pausas y entonación. La **latencia** es el tiempo entre la
solicitud y el inicio o fin de la respuesta.

La selección de voz, prosodia, latencia, consentimiento y riesgo de
suplantación requieren un tratamiento específico que queda para el
Track Salud.

**Pregunta de control:** si Whisper transcribe correctamente “dolor de
pecho”, ¿ya tomó una decisión clínica? No. ASR produjo texto; cualquier
interpretación o acción pertenece a etapas posteriores y necesita sus
propias validaciones.

---
## 4. Del texto a los tokens

**NLP** (*Natural Language Processing*) o **PLN** (Procesamiento de
Lenguaje Natural) es el área que aplica métodos computacionales al
texto y al habla. ASR termina en texto; desde ese punto podemos aplicar
tareas de NLP.

Un modelo no recibe directamente “palabras con significado”. Un
**tokenizador** es un componente determinista que:

1. divide el texto en unidades llamadas tokens;
2. consulta un vocabulario;
3. reemplaza cada token por un identificador entero.

```text
"La médica revisó."
        ↓ tokenización
["La", "médica", "revisó", "."]
        ↓ vocabulario
[41, 927, 3051, 13]
```

Los números del ejemplo son identificadores, no puntajes. Un ID más
alto no significa que un token sea más importante o más frecuente.

Un token puede ser:

- una palabra completa;
- una parte de palabra;
- un signo de puntuación;
- un símbolo especial.

El separador simple de abajo sirve para visualizar la idea. Los LLMs
usan tokenizadores de **subpalabras**, que pueden dividir una palabra
infrecuente en piezas reutilizables. Por ejemplo,
`reprogramación → re + program + ación` sería una división ilustrativa;
cada tokenizador real puede elegir piezas distintas.

Por eso:

- cantidad de palabras y cantidad de tokens no son equivalentes;
- un cambio pequeño de texto puede cambiar varios tokens;
- la ventana de contexto se mide en tokens, no en páginas ni palabras.

In [ ]:
texto = "La médica revisó rápidamente el informe clínico."
tokens_didacticos = re.findall(r"\w+|[^\w\s]", texto, flags=re.UNICODE)
vocabulario = {
    token: indice
    for indice, token in enumerate(dict.fromkeys(tokens_didacticos))
}
ids = [vocabulario[token] for token in tokens_didacticos]

display(
    pd.DataFrame(
        {
            "posición": range(len(tokens_didacticos)),
            "token didáctico": tokens_didacticos,
            "id local": ids,
        }
    )
)
print("Texto original:", texto)
print("Secuencia numérica:", ids)

### Mapa mínimo de NLP

Una **tarea** define qué salida esperamos del sistema. Una **entidad**
es un fragmento identificado como perteneciente a un tipo, por ejemplo
una fecha, un medicamento o una organización.

| Tarea | Pregunta que responde | Salida de ejemplo |
|---|---|---|
| Clasificación | ¿A qué categoría pertenece el texto? | `reclamo` |
| NER | ¿Qué entidades aparecen? | `MEDICAMENTO: ibuprofeno` |
| Embeddings | ¿Qué textos tienen significado parecido? | vector |
| Generación | ¿Qué token podría continuar? | texto nuevo |
| Traducción | ¿Cómo expresar esto en otro idioma? | otro texto |

**NER** significa *Named Entity Recognition*. Localiza fragmentos y
les asigna tipos. En salud, que algo tenga apariencia de entidad no
garantiza que sea correcto: abreviaturas, negaciones y contexto
clínico vuelven la tarea más delicada.

Los pipelines comparten el inicio, pero divergen en la salida:

```text
texto → tokens → modelo
                  ├── clasificación → una categoría para el texto
                  ├── NER           → fragmentos + tipos
                  ├── embedding     → un vector
                  └── generación    → nuevos tokens
```

El tokenizador no resuelve ninguna de estas tareas por sí solo. Solo
prepara la representación de entrada que el modelo espera.

### De ID a embedding

El identificador entero sirve para consultar una tabla aprendida. La
fila correspondiente es el embedding inicial del token:

```text
token "médica" → ID 927 → fila 927 de la tabla → vector numérico
```

Tokens iguales comienzan con el mismo vector, pero el transformer
construye representaciones **contextuales**: después de attention, la
representación de una palabra puede cambiar según las palabras que la
rodean.

---
## 5. Transformers y attention

Un **transformer** es una arquitectura de red neuronal para procesar
secuencias. Una secuencia es una colección ordenada: en texto, una
secuencia de tokens; en audio, una secuencia de tramos o features.

El recorrido simplificado es:

```text
tokens
  ↓ tokenización e IDs
embeddings iniciales + información de posición
  ↓
bloque transformer 1
  ├── attention
  └── red feed-forward
  ↓
bloque transformer 2
  ↓
 ...
  ↓
representaciones contextuales
  ↓
salida de la tarea
```

La **información de posición** permite distinguir órdenes como
“modelo usa datos” y “datos usa modelo”. Attention por sí sola compara
contenido; el orden debe incorporarse de alguna forma.

### Query, Key y Value

Para cada token, el bloque calcula tres vectores diferentes mediante
transformaciones aprendidas:

| Vector | Pregunta intuitiva | Analogía técnica sobria |
|---|---|---|
| **Query (Q)** | ¿Qué información busca esta posición? | Criterio de una consulta |
| **Key (K)** | ¿Qué información ofrece cada posición? | Metadatos usados para comparar |
| **Value (V)** | ¿Qué contenido aporta la posición? | Payload que se recupera |

La analogía se parece a una búsqueda en memoria: se compara una
consulta con claves y se combinan los contenidos asociados. La
diferencia importante es que Q, K y V no fueron escritos por una
persona ni son campos de una base de datos. Son vectores calculados
por el modelo a partir de parámetros aprendidos.

Attention ejecuta cuatro pasos:

1. compara cada Query con todas las Keys;
2. obtiene un **score** o puntaje de compatibilidad;
3. `softmax` convierte los scores de cada fila en pesos positivos que
   suman 1;
4. usa esos pesos para calcular una suma ponderada de los Values.

```text
Query de "clínicos"
         ↓ comparar
Keys de ["El", "modelo", "analiza", "datos", "clínicos"]
         ↓ pesos
Values combinados → nueva representación contextual de "clínicos"
```

En la fórmula, `d` es la dimensión de los vectores. Dividir por
`√d` evita que los scores crezcan demasiado cuando hay muchas
dimensiones.

$$Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d}}\right)V$$

No hace falta memorizar la fórmula. Lo importante es el recorrido:
**comparar → asignar pesos → combinar información**.

La próxima práctica usa embeddings inventados de cuatro dimensiones.
No son embeddings producidos por un modelo real: permiten inspeccionar
todos los números sin ocultar el mecanismo.

In [ ]:
# Embeddings didácticos: cada columna representa una propiedad
# inventada para poder observar el mecanismo.
tokens_attention = ["El", "modelo", "analiza", "datos", "clínicos"]
dimensiones = ["agente", "acción", "objeto", "dominio_salud"]
embeddings = np.array(
    [
        [0.05, 0.00, 0.00, 0.00],  # El
        [1.00, 0.20, 0.10, 0.20],  # modelo
        [0.50, 1.00, 0.60, 0.10],  # analiza
        [0.10, 0.20, 1.00, 0.45],  # datos
        [0.00, 0.00, 0.55, 1.00],  # clínicos
    ],
    dtype=float,
)

display(
    pd.DataFrame(
        embeddings,
        index=tokens_attention,
        columns=dimensiones,
    )
)

In [ ]:
def softmax(filas):
    filas = filas - filas.max(axis=-1, keepdims=True)
    exp = np.exp(filas)
    return exp / exp.sum(axis=-1, keepdims=True)


# En esta demo Q y K comparten la misma representación para que la
# relación sea visible. En un transformer real son proyecciones
# diferentes y aprendidas.
Q = embeddings.copy()
K = embeddings.copy()
V = embeddings.copy()

scores = Q @ K.T / np.sqrt(Q.shape[1])
pesos_attention = softmax(scores)
salida_contextual = pesos_attention @ V

fig, ax = plt.subplots(figsize=(7.5, 6))
mapa = ax.imshow(pesos_attention, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(tokens_attention)), tokens_attention, rotation=35)
ax.set_yticks(range(len(tokens_attention)), tokens_attention)
ax.set_xlabel("Keys: tokens que pueden aportar información")
ax.set_ylabel("Queries: token que busca información")
ax.set_title("Pesos de attention — ejemplo didáctico")

for fila in range(len(tokens_attention)):
    for columna in range(len(tokens_attention)):
        ax.text(
            columna,
            fila,
            f"{pesos_attention[fila, columna]:.2f}",
            ha="center",
            va="center",
            fontsize=8,
        )
fig.colorbar(mapa, ax=ax, label="peso")
plt.tight_layout()
plt.show()

print(
    "Cada fila suma:",
    np.round(pesos_attention.sum(axis=1), 6),
)

### Qué observar

En el mapa de calor:

- cada **fila** corresponde a la Query de un token;
- cada **columna** corresponde a la Key de un token;
- cada celda indica cuánto Value de esa columna se incorpora a la
  nueva representación de la fila.

- Cada fila suma 1 porque `softmax` crea una distribución de pesos.
- Un peso alto significa que esa posición aporta más a la mezcla,
  no que el modelo haya explicado una relación causal.
- La salida de cada token cambia porque incorpora valores de otros
  tokens.

Un mapa de attention es una señal interna, no una explicación completa
de por qué el modelo respondió algo.

### Qué más contiene el bloque

| Componente | Función introductoria |
|---|---|
| Multi-head attention | Repite attention en paralelo con distintas proyecciones |
| Cabeza | Una instancia de Q, K y V con sus propios parámetros |
| Red feed-forward | Transforma cada posición después de combinar contexto |
| Conexión residual | Suma la entrada previa para no perder toda su información |
| Normalización | Mantiene escalas numéricas más estables entre bloques |

Varias cabezas permiten capturar patrones distintos al mismo tiempo,
pero una cabeza no tiene una etiqueta humana fija como “gramática” o
“medicina”. Cualquier especialización debe analizarse; no se presupone.

```text
entrada del bloque
     ├───────────────┐
     ↓               │
multi-head attention │
     ↓               │
  suma residual ◀────┘
     ↓
normalización
     ├───────────────┐
     ↓               │
feed-forward          │
     ↓               │
  suma residual ◀────┘
     ↓
normalización → salida del bloque
```

---
## 6. Contexto, generación autoregresiva y KV cache

Un **LLM** es un modelo de lenguaje con una gran cantidad de
parámetros, entrenado para trabajar con secuencias de tokens. En un
LLM generativo, la tarea básica es estimar una distribución de
probabilidades para el próximo token.

**Autoregresivo** significa que cada nueva predicción usa como entrada
los tokens anteriores, incluidos los que el propio modelo acaba de
generar.

```text
"El informe"
     ↓ predicción
"indica"
     ↓ agregar al contexto y predecir otra vez
"que"
     ↓
...
```

En cada paso, el modelo:

1. procesa los tokens disponibles;
2. calcula una distribución para el siguiente token;
3. selecciona un token mediante una estrategia de decodificación;
4. lo agrega al contexto y repite.

La **decodificación** define cómo elegir desde la distribución. Elegir
siempre el token más probable produce una salida determinista; el
muestreo puede introducir variación. En esta clase no compararemos
estrategias de decodificación.

### Tres cosas que suelen confundirse

| Elemento | Qué contiene | Cuándo cambia |
|---|---|---|
| Pesos del modelo | Patrones aprendidos durante entrenamiento | Cuando se entrena o ajusta el modelo |
| Contexto | Tokens del prompt y de la respuesta actual | Durante cada solicitud |
| KV cache | Keys y Values calculados para ese contexto | Mientras avanza la generación |

La **ventana de contexto** limita cuántos tokens puede considerar el
modelo en una solicitud. Tener un documento dentro de esa ventana no
modifica los pesos. La KV cache tampoco es memoria de largo plazo:
normalmente se descarta cuando termina la generación.

### Prefill y decodificación

La generación puede separarse en dos etapas:

```text
prompt completo
     ↓ PREFILL: procesar todos los tokens y crear K/V
KV cache inicial
     ↓ DECODIFICACIÓN: generar un token
agregar K/V del token nuevo
     ↓ DECODIFICACIÓN: generar el siguiente
...
```

Sin cache, cada paso volvería a calcular K y V para posiciones ya
procesadas:

```text
sin cache: [prompt + token 1 + ...] → recalcular todo
con cache: [K/V anteriores guardados] + calcular solo K/V nuevo
```

Guardar resultados evita trabajo repetido, pero ocupa memoria. La
memoria crece aproximadamente con:

```text
tokens almacenados × capas × cabezas × dimensión por cabeza
                   × 2 (K y V) × bytes por valor
```

- **capa:** un bloque transformer dentro del modelo;
- **cabeza:** una instancia paralela de attention;
- **dimensión por cabeza:** cantidad de valores de cada vector K o V;
- **bytes por valor:** memoria usada por cada número según su tipo.

> La KV cache evita trabajo repetido, pero no vuelve constante todo
> el costo de attention: el token nuevo todavía debe compararse con
> el contexto previo.

La simulación siguiente cuenta “unidades de trabajo” para mostrar la
tendencia. No ejecuta un transformer ni mide milisegundos reales.

In [ ]:
def simular_kv_cache(
    tokens_prompt,
    tokens_nuevos,
    capas=24,
    cabezas=16,
    dimension_cabeza=64,
    bytes_por_valor=2,
):
    pasos = np.arange(1, tokens_nuevos + 1)

    # Unidades didácticas de proyecciones K/V acumuladas.
    trabajo_sin_cache = np.cumsum(tokens_prompt + pasos - 1)
    trabajo_con_cache = tokens_prompt + pasos

    tokens_almacenados = tokens_prompt + pasos
    bytes_cache = (
        2  # K y V
        * capas
        * cabezas
        * dimension_cabeza
        * tokens_almacenados
        * bytes_por_valor
    )
    cache_mb = bytes_cache / 1024 ** 2

    return pd.DataFrame(
        {
            "token_generado": pasos,
            "trabajo_sin_cache": trabajo_sin_cache,
            "trabajo_con_cache": trabajo_con_cache,
            "cache_MB": cache_mb,
        }
    )


simulacion = simular_kv_cache(
    tokens_prompt=64,
    tokens_nuevos=40,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(
    simulacion["token_generado"],
    simulacion["trabajo_sin_cache"],
    label="sin KV cache",
    linewidth=2,
)
axes[0].plot(
    simulacion["token_generado"],
    simulacion["trabajo_con_cache"],
    label="con KV cache",
    linewidth=2,
)
axes[0].set(
    title="Trabajo acumulado de proyección K/V",
    xlabel="Tokens nuevos generados",
    ylabel="Unidades didácticas de trabajo",
)
axes[0].legend()
axes[0].grid(alpha=0.2)

axes[1].plot(
    simulacion["token_generado"],
    simulacion["cache_MB"],
    color="#dc2626",
    linewidth=2,
)
axes[1].set(
    title="Memoria ocupada por la KV cache",
    xlabel="Tokens nuevos generados",
    ylabel="MB (configuración hipotética)",
)
axes[1].grid(alpha=0.2)
plt.tight_layout()
plt.show()

display(simulacion.iloc[[0, 9, 19, -1]].round(2))

### Interpretación correcta

- La línea “sin cache” crece rápido porque vuelve a proyectar tokens
  anteriores en cada paso.
- La línea “con cache” representa proyecciones K/V reutilizadas; no
  representa todo el cómputo del modelo.
- La cache acelera la etapa de decodificación a cambio de memoria.
- Más contexto, más capas, más cabezas o mayor precisión numérica
  requieren más memoria.
- La gráfica es una **simulación conceptual**, no un benchmark de un
  modelo concreto.

### Ejemplo de decisión de ingeniería

Un servidor puede atender más solicitudes simultáneas si cada una usa
menos memoria de cache. Reducir el contexto ahorra memoria, pero puede
eliminar información necesaria para responder. No existe un máximo
universal: la decisión depende de la tarea, el modelo, el hardware y
la cantidad de solicitudes concurrentes.

**Pregunta de control:** ¿la KV cache hace que el modelo “recuerde” una
conversación mañana? No. Reutiliza cálculos de la generación actual;
persistir una conversación es responsabilidad de otra parte del
sistema.

---
## 7. Actividad integradora — 20 minutos

La actividad cambia entradas de experimentos ya construidos. No
implementes ASR, attention ni la KV cache desde cero.

| Parte | Tiempo | Cambio | Evidencia esperada |
|---|---:|---|---|
| Audio | 6 min | Inicio y fin del tramo | Waveform, espectrograma y RMS |
| Attention | 7 min | Token y embedding didáctico | Dos mapas de calor |
| KV cache | 7 min | Contexto o tokens generados | Tabla con trabajo y MB |

Modificá únicamente las variables marcadas con `TODO`.

### Parte 1 — seleccionar un tramo

Elegí valores entre `0` y la duración total. Un tramo corto puede
contener una pausa, parte de una palabra o varias sílabas. Compará su
RMS con el valor del audio completo.

### Parte 2 — alterar el ejemplo de attention

Las cuatro dimensiones son propiedades inventadas para la
demostración. Ajustarlas no “entrena” el modelo. Sirve para comprobar
que cambiar una representación modifica las comparaciones Q–K y, por
lo tanto, varios pesos.

### Parte 3 — comparar contextos

Mantené constantes las capas, cabezas y precisión. Cambiá una sola
variable por vez para poder atribuir el efecto observado.

In [ ]:
# TODO 1: elegí un tramo dentro de la duración del audio.
INICIO_SEGUNDOS = 0.5
FIN_SEGUNDOS = min(2.0, duracion)

inicio_muestra = int(INICIO_SEGUNDOS * sample_rate)
fin_muestra = int(FIN_SEGUNDOS * sample_rate)
tramo = audio[inicio_muestra:fin_muestra]

if len(tramo) == 0:
    raise ValueError("El tramo está vacío; revisá inicio y fin.")

f_tramo, t_tramo, p_tramo = spectrogram(
    tramo,
    fs=sample_rate,
    nperseg=min(400, len(tramo)),
    noverlap=min(240, max(0, len(tramo) // 2)),
    nfft=512,
)

fig, axes = plt.subplots(2, 1, figsize=(11, 6))
axes[0].plot(
    np.arange(len(tramo)) / sample_rate + INICIO_SEGUNDOS,
    tramo,
    linewidth=0.7,
)
axes[0].set(title="Waveform del tramo", ylabel="Amplitud")
axes[1].pcolormesh(
    t_tramo + INICIO_SEGUNDOS,
    f_tramo,
    10 * np.log10(p_tramo + 1e-10),
    shading="auto",
    cmap="magma",
)
axes[1].set(
    title="Espectrograma del tramo",
    xlabel="Tiempo (s)",
    ylabel="Frecuencia (Hz)",
    ylim=(0, 8000),
)
plt.tight_layout()
plt.show()

print("Duración del tramo:", round(len(tramo) / sample_rate, 2), "s")
print("Energía RMS:", round(float(np.sqrt(np.mean(tramo ** 2))), 4))

In [ ]:
# TODO 2: reemplazá "clínicos" y ajustá sus cuatro propiedades.
NUEVO_TOKEN = "satelitales"
EMBEDDING_NUEVO = np.array([0.0, 0.0, 0.65, 0.85])

tokens_modificados = tokens_attention[:-1] + [NUEVO_TOKEN]
embeddings_modificados = embeddings.copy()
embeddings_modificados[-1] = EMBEDDING_NUEVO
pesos_modificados = softmax(
    embeddings_modificados @ embeddings_modificados.T
    / np.sqrt(embeddings_modificados.shape[1])
)

cambio_promedio = np.abs(
    pesos_modificados - pesos_attention
).mean()
print("Tokens:", tokens_modificados)
print(f"Cambio promedio de los pesos: {cambio_promedio:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, matriz, titulo, etiquetas in [
    (axes[0], pesos_attention, "Antes", tokens_attention),
    (axes[1], pesos_modificados, "Después", tokens_modificados),
]:
    ax.imshow(matriz, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(etiquetas)), etiquetas, rotation=35)
    ax.set_yticks(range(len(etiquetas)), etiquetas)
    ax.set_title(titulo)
plt.tight_layout()
plt.show()

In [ ]:
# TODO 3: cambiá el contexto o los tokens a generar.
CONTEXTO_CORTO = 64
CONTEXTO_LARGO = 512
TOKENS_A_GENERAR = 40

comparacion = []
for nombre, contexto in [
    ("contexto corto", CONTEXTO_CORTO),
    ("contexto largo", CONTEXTO_LARGO),
]:
    resultado = simular_kv_cache(contexto, TOKENS_A_GENERAR).iloc[-1]
    comparacion.append(
        {
            "escenario": nombre,
            "tokens_prompt": contexto,
            "trabajo_sin_cache": resultado["trabajo_sin_cache"],
            "trabajo_con_cache": resultado["trabajo_con_cache"],
            "cache_final_MB": resultado["cache_MB"],
        }
    )

display(pd.DataFrame(comparacion).round(2))

### Entregable breve

Escribí cuatro observaciones de una o dos frases:

1. ¿Qué diferencia visible encontraste entre waveform y
   espectrograma?
2. ¿La transcripción provino de Whisper o del fallback? ¿Cómo lo
   comprobaste?
3. ¿Por qué modificar un token puede cambiar las representaciones de
   otros tokens?
4. ¿Qué ganamos y qué pagamos al usar KV cache con contexto largo?

**Criterio de aceptación:** las respuestas deben citar al menos un
valor o gráfico producido por el notebook.

| La respuesta está completa si… | Ejemplo de evidencia |
|---|---|
| Distingue tiempo de frecuencia | “El pico aparece cerca de 1,2 s y la energía ocupa varias bandas” |
| Identifica el modo de ASR | La línea `Modo:` indica Whisper o fallback |
| Relaciona representación y pesos | El cambio promedio es distinto de cero |
| Explica el intercambio de KV cache | Compara MB y trabajo entre ambos contextos |

No se evalúa obtener “el gráfico más lindo”. Se evalúa interpretar qué
representa cada salida y qué limitación tiene.

---

## Síntesis de la clase

- Una waveform representa amplitud en el tiempo; un espectrograma agrega información de frecuencias.
- ASR transforma audio en texto y TTS recorre el camino inverso.
- Tokenizar es convertir texto en unidades e identificadores procesables.
- Attention compara Query con Keys y combina Values según sus pesos.
- Un LLM genera tokens de manera autoregresiva dentro de una ventana de contexto.
- La KV cache evita recalcular K y V anteriores, mejora velocidad y consume memoria creciente.

## Comprobación conceptual

Antes de continuar, intentá responder sin mirar el notebook:

1. ¿Cuál era el problema central de la clase?
2. ¿Qué entrada recibió el sistema y qué salida produjo?
3. ¿Qué decisión humana siguió siendo necesaria?
4. ¿Qué limitación observaste en el experimento?

Si podés explicarlo con tus propias palabras y justificarlo con un resultado visible, alcanzaste el objetivo introductorio.

## Puente con la próxima clase

En la clase 7 usaremos embeddings y un LLM local para construir un mini-RAG, producir JSON y simular una herramienta de agente.

## Conexión con los tracks

El Track Salud profundizará ASR/TTS, NLP clínico, transformers, prompting, RAG y agentes. Esta clase deja el vocabulario y las intuiciones necesarias para trabajar esos temas sin tratarlos todavía como una caja negra.

La implementación profunda, el trabajo con datasets reales y las decisiones de producción se desarrollarán en los módulos especializados.